In [ ]:
df = df.dropna(subset=["review_text", "overall_rating"]).copy()
df["review_text"] = df["review_text"].astype(str).str.strip()
df = df[df["review_text"].str.len() > 0]

df["sentimento"] = np.select(
    [
        df["overall_rating"].str.contains(['atrasou', 'esta atrasado', 'demorou', 'nao chegou']),
        df["overall_rating"].str.contains(['no prazo', 'chegou antes']),
    ],
    ["atrasado", "no prazo"],
    default="neutro",
)

# Exclui avaliações neutras neste primeiro experimento.
df = df[df["sentimento"] != "neutro"].copy()

MAX_EXAMPLES = 20000  # experimente 5_000, 20_000, 50_000 ou None
if MAX_EXAMPLES is not None and len(df) > MAX_EXAMPLES:
    # Amostragem estratificada mantém aproximadamente a proporção das classes.
    df = (
        df.groupby("sentimento", group_keys=False)
            .apply(
                lambda grupo: grupo.sample(
                    n=max(1, round(MAX_EXAMPLES * len(grupo) / len(df))),
                    random_state=RANDOM_STATE,
                )
            )
            .sample(frac=1, random_state=RANDOM_STATE)
            .reset_index(drop=True)
    )

print(f"Exemplos após preparação: {len(df):,}")
display(df[["review_text", "overall_rating", "sentimento"]].head())